## Main Objective: 
Build Streamlit dashboard that predicts:
- Likelihood of sequence disruption using operational features
- (further) severity of disruption
- provides simple operational reccomendation




## Load Libraries & Data

In [2]:
## Load Libraties and Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

Importing libraries needed for data manipulation, visualization,splitting the data, and building a baseline classification model.

In [3]:
#Loading Data & Minor Cleansing
seq_spoil = pd.read_csv("../data/raw/sequence_spoilage.csv", sep="|")
assign_history = pd.read_csv("../data/raw/Assignment_History.csv")

seq_spoil = seq_spoil.loc[:, ~seq_spoil.columns.str.contains("^Unnamed")]
assign_history = assign_history.loc[:, ~assign_history.columns.str.contains("^Unnamed")]

seq_spoil.columns = seq_spoil.columns.str.upper().str.strip()
assign_history.columns = assign_history.columns.str.upper().str.strip()

print("Sequence spoilage shape:", seq_spoil.shape)
print("Assignment history shape:", assign_history.shape)

Sequence spoilage shape: (9025, 22)
Assignment history shape: (57067, 43)


/var/folders/lc/6jh43b5s4hz2fn996qp8hr2h0000gn/T/ipykernel_58433/578813477.py:3: DtypeWarning: Columns (32,38) have mixed types. Specify dtype option on import or set low_memory=False.
  assign_history = pd.read_csv("../data/raw/Assignment_History.csv")


Loaded both datasets and removed unnamed columns that came from the file export. I also standardized the column names by making them uppercase and stripping extra spaces so I could work with them more consistently later.

The sequence spoilage dataset is much smaller and already closer to sequence level, while the assignment history dataset contains many more rows and looks more like event or log data.

## Exploration of Data

In [4]:
seq_spoil.head()
assign_history.head()

,CONTRCT_YR,CONTRCT_MONTH,FLEET_CD,SEQ_NBR,SEQ_ORIGIN_DT,SEQ_BASE,IS_BLOCK_OE_IND,SEQ_OE_SNPSHT_ID,SEQ_ID,SEQ_SOURCE_CD,...,SEQ_EST_DEP_TMS,SEQ_EST_ARVL_TMS,SEQ_ACTL_DEP_TMS,SEQ_ACTL_ARVL_TMS,CKP_ASSIGNED_IND,STUDENT_ASSIGNED_IND,SEQ_ACTIVE_IND,TRIP_ACTIVE_IND,LOAD_TMS,SF_LOAD_TMS
0,2024,11,320,4635,11/17/2024,CLT,T,00686803_320_NOV2024_1_2024100204075507709,94258.0,FSA,...,2024-11-17T07:31:00.000-06:00,2024-11-18T16:09:00.000-06:00,2024-11-17T07:31:00.000-06:00,2024-11-18T16:09:00.000-06:00,TRUE,False,False,False,2025-09-11T13:44:03.620-05:00,2026-02-19 22:11:37.711
1,2024,11,320,4635,11/17/2024,CLT,T,00686803_320_NOV2024_1_2024100204075507709,94258.0,FSA,...,2024-11-17T07:31:00.000-06:00,2024-11-18T16:09:00.000-06:00,2024-11-17T07:31:00.000-06:00,2024-11-18T16:09:00.000-06:00,TRUE,False,False,False,2025-09-11T13:44:03.620-05:00,2026-02-19 22:11:37.711
2,2024,11,320,4635,11/17/2024,CLT,T,00686803_320_NOV2024_1_2024100204075507709,94258.0,FSA,...,2024-11-17T07:31:00.000-06:00,2024-11-18T16:09:00.000-06:00,2024-11-17T07:31:00.000-06:00,2024-11-18T16:09:00.000-06:00,TRUE,False,False,False,2025-09-11T13:44:03.620-05:00,2026-02-19 22:11:37.711
3,2024,11,320,4635,11/17/2024,CLT,T,00686803_320_NOV2024_1_2024100204075507709,94258.0,FSA,...,2024-11-17T07:31:00.000-06:00,2024-11-18T16:09:00.000-06:00,2024-11-17T07:31:00.000-06:00,2024-11-18T16:09:00.000-06:00,TRUE,False,False,False,2025-09-11T13:44:03.620-05:00,2026-02-19 22:11:37.711
4,2024,11,320,4635,11/17/2024,CLT,T,00686803_320_NOV2024_1_2024100204075507709,94258.0,FSA,...,2024-11-17T07:31:00.000-06:00,2024-11-18T16:09:00.000-06:00,2024-11-17T07:31:00.000-06:00,2024-11-18T16:09:00.000-06:00,TRUE,False,False,False,2025-09-11T13:44:03.620-05:00,2026-02-19 22:11:37.711


Looked at the first few rows of the seq_spoil to understand what each row represents and features availible and the sequence level. While the assign_history dataset has more detailed operational records, with multiple rows per sequence.


In [5]:
seq_spoil.isna().sum().sort_values(ascending=False).head(10)

TOTAL_SPOILED_HRS    6306
TOTAL_BLOCKED_HRS    4605
SEQ_NBR                 0
MAX_LEGS_PER_DAY        0
SEQ_START_HRS           0
FLIGHT_PATTERN          0
SEQ_START               0
SEQ_PATTERN             0
LAYOVER                 0
IN_SEQ_DHD              0
dtype: int64

Missing value check allows us to see which columns will need cleaning later, at this point I am looking for missing values that can impact modeling.

In [6]:
assign_history.isna().sum().sort_values(ascending=False).head(43)

STUDENT_OE_START_DT      40669
STUDENT_OE_END_DT        40669
STUDENT_TRNG_TYPE        40302
STUDENT_BASE             40197
IS_STUDENT_OE_STUDENT    40197
STUDENT_POSITN_CD        40197
STUDENT_EMP_NBR          40197
TRIP_COORD_CMNT_TXT      33492
TRIP_SOURCE_CD           28962
CKP_ROLE_TYPE_DESC       17807
CKP_EMP_NBR              17082
CKP_BASE                 17082
SEQ_OE_SNPSHT_ID         10617
FOS_SEQ_STATUS_TXT        7830
TRIP_QLA_MSG_JSON_TXT     6155
TRIP_ID                   1451
TRIP_ROW_EFF_TMS          1451
TRIP_ROW_EXPIRY_TMS       1451
SEQ_DUTY_PERIOD_CT        1446
SEQ_CALNDR_DAY_CT         1446
SEQ_EST_DEP_TMS           1446
SEQ_EST_ARVL_TMS          1446
SEQ_ACTL_DEP_TMS          1446
STATUS_TXT                1446
SEQ_POSITN_CD             1446
SEQ_ROW_EXPIRY_TMS        1446
SEQ_ROW_EFF_TMS           1446
SEQ_SOURCE_CD             1446
SEQ_ID                    1446
IS_BLOCK_OE_IND           1446
SEQ_BASE                  1446
SEQ_ACTL_ARVL_TMS         1446
STUDENT_

assign_history dataset has so many missing values which can cause issues with modeling. So I decided to omit it for now to not be used as the main modeling table for the first version of the project.

In [10]:
seq_spoil.to_csv('seq_spoil.csv', index=False)
assign_history.to_csv('assign_history.csv', index=False)

## Building a Clean Dataset with Feature Considerations for Dashboard

In [63]:
seq_spoil_clean = seq_spoil.groupby("SEQ_NBR", as_index=False).agg({
    "SPOILAGE": "first",
    "TOTAL_BLOCKED_HRS": "mean",
    "SEQ_TTL_LEGS": "max",
    "LAYOVER": "mean",
    "SEQ_START_HRS": "first"
})

print(seq_spoil_clean.shape)
seq_spoil_clean.head()

(5207, 6)


,SEQ_NBR,SPOILAGE,TOTAL_BLOCKED_HRS,SEQ_TTL_LEGS,LAYOVER,SEQ_START_HRS
0,101,NOT SPOILED,4.30200,2,6.000,17
1,102,NOT SPOILED,4.27125,2,13.125,17
2,103,NOT SPOILED,NaN,2,13.000,17
3,105,PARTIALLY SPOILED,NaN,3,12.000,18
4,108,NOT SPOILED,NaN,2,25.000,16


Here I grouped seq_spoil to one row per sequence using "SEQ_NBR" as the join key. For each sequence, I kept the spoilage label and a few features I thought we useful for prediction.
I chose:
- TOTAL_BLOCKED_HRS
- SEQ_TTL_LEGS
- LAYOVER
- SEQ_START_HRS

because they can help show the relation of how demanding or complex a sequence may be.


## Define the Target Goal

In [64]:
print(seq_spoil_clean["SPOILAGE"].value_counts(dropna=False))

SPOILAGE
NOT SPOILED          3408
FULLY SPOILED        1003
PARTIALLY SPOILED     796
Name: count, dtype: int64


Before modeling, I checked the distribution of Spoilage categories across the df to see if the target was balanced and if it would make sense to simplify it for the first model.

In [65]:
seq_spoil_clean["RISK_FLAG"] = seq_spoil_clean["SPOILAGE"].apply(
    lambda x: 0 if x == "NOT SPOILED" else 1
)

print(seq_spoil_clean["RISK_FLAG"].value_counts())

RISK_FLAG
0    3408
1    1799
Name: count, dtype: int64


I decided to convert spoilage into a binary flag becuase binary classification is easier to interpret and works well for the dashboard goal of showing low risk vs. elevated risk
With 0 = not spoiled & 1 = partially/fully spoiled

## Define Features & Target Gaol

In [66]:
X = seq_spoil_clean[[
    "TOTAL_BLOCKED_HRS",
    "SEQ_TTL_LEGS",
    "LAYOVER",
    "SEQ_START_HRS"
]]

y = seq_spoil_clean["RISK_FLAG"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (5207, 4)
Target shape: (5207,)


At this stage, I defined the input features and the target variable. I wanted to keep the features minimal to iunderstand the model better and make it easier to turn into dashboard inputs.

In [67]:
print(X.isna().sum().sort_values(ascending=False))
X = seq_spoil_clean[feature_cols].copy()
y = seq_spoil_clean["RISK_FLAG"].copy()

# fill missing numeric values with median
X = X.fillna(X.median(numeric_only=True))

print(X.isna().sum())

TOTAL_BLOCKED_HRS    2253
SEQ_TTL_LEGS            0
LAYOVER                 0
SEQ_START_HRS           0
dtype: int64
TOTAL_BLOCKED_HRS    0
SEQ_TTL_LEGS         0
LAYOVER              0
SEQ_START_HRS        0
dtype: int64


I ran into an error because logistic regression cannot train on missing values. To fix this, I filled missing numeric feature values with the median of each column. I chose the median because it is more stable than the mean when features may contain outliers.

## Train-Test Split

In [68]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (4165, 4)
X_test shape: (1042, 4)


I split the data into train and test sets. I used stratifciation so the class distribution would stay similar accross both sets, which is important when target isn't perfectly balance.

## Train Baseline Model

In [69]:
model = LogisticRegression(class_weight="balanced",max_iter=1000)
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

I chose logistic regression as a baseline model becuase it is simple interpretable, and a good first check for whether the features can aid in prediction. I used class_weight= "balanced" so the model would pay more attnetion to the minority instead of defaulting to heavily to the majority.

## Evaluate the Model

In [70]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.71      0.53      0.61       682
           1       0.40      0.59      0.48       360

    accuracy                           0.55      1042
   macro avg       0.56      0.56      0.54      1042
weighted avg       0.60      0.55      0.56      1042

[[361 321]
 [146 214]]


This report shows how the model distinguishes between low-risk and elevated-risk sequences. I am paying the most attention to recall for the elevated-risk class, becuase in an operational setting it is more important to catch risky sequences than to optimize solely overall accuracy. 

The model shows recall of class 1 being 0.62 so it catches approx 62% of risky sequences. However, we are currently over-flagging risk. It shows precision of class 1 to be 0.35 which shows that only 35% of flagged sequences are truly risky while 65% are false alarms. Which explains our confusion matrix value of 283 false positives

Overall, the model is more aggressive in predicting risk, prioritizing detection over precision. This behavior is reasonable for an operational setting where it is more important to catch potential disruptions than to avoid extra warnings.

## Random Forest Model 

In [71]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.68      0.71      0.69       682
           1       0.39      0.36      0.38       360

    accuracy                           0.59      1042
   macro avg       0.53      0.53      0.53      1042
weighted avg       0.58      0.59      0.58      1042



I tried to compare logistic regression and random forest models for predicting disruption risk.The random forest one attained higher overall accuracy (0.66 vs 0.55), but performed significantly worse at identifying elevated risk sequences. Its recall for the risk class was only 0.24, meaning it missed majority of the true risk. 
However, in contrast, logistic regression achieved a higher recall of 0.62 for the risk class, making it a more effect model for predicting sequences that may result in disruption.
Although logistic regression produces more false positives, it is better aligned with the operational goal of identifying potential disruptions early. Missing a disruption is more costly than issuing extra warnings, so I selected logistic regression as the final model.


## Finetuning the Threshold

In [72]:
# get probabilities instead of hard predictions
probs = model.predict_proba(X_test)[:, 1]

# apply custom threshold (instead of default 0.5)
y_pred = (probs > 0.6).astype(int)

# evaluate
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.68      0.87      0.76       682
           1       0.47      0.22      0.30       360

    accuracy                           0.64      1042
   macro avg       0.57      0.54      0.53      1042
weighted avg       0.61      0.64      0.60      1042

[[591  91]
 [280  80]]


I experimented with adjusting the classification threshold to control the tradeoff between precision and recall. At a higher threshold (0.6), the model became more conservative, reducing false positives and increasing precision. However, recall for the elevated-risk class dropped significantly to 0.21, meaning most disruptions were missed. At the default threshold (0.5), the model achieved a much higher recall of 0.62, capturing the majority of risky sequences. 

Although this resulted in more false positives, it better aligns with the operational goal of identifying potential disruptions early.

Based on this tradeoff, I selected the default threshold of 0.5 for the final model,prioritizing recall over precision.

In [73]:
import joblib

# define feature order from your training data
feature_cols = X.columns.tolist()

# save both
joblib.dump(model, "logistic_spoilage_model.pkl")
joblib.dump(feature_cols, "feature_cols.pkl")

print("Saved successfully")

Saved successfully
